In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


In [64]:
#1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
def clean_city_robust(s):
    s_clean = s.strip().lower()
    # Наш словарь сопоставления
    mapping = {
        "almaty": "Алматы", "алматы": "Алматы",
        "astana": "Астана", "астана": "Астана",
        "shymkent": "Шымкент", "шымкент": "Шымкент",
        "karaganda": "Караганда", "караганда": "Караганда",
        "aktobe": "Актобе", "актобе": "Актобе",
        "atyrau": "Атырау", "атырау": "Атырау"
    }
    return mapping.get(s_clean, s.strip().capitalize())

In [ ]:
#2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
category_count = {}
for order in rows:
    cat = order["category"]
    category_count[cat] = category_count.get(cat, 0) + 1
for cat, count in sorted(category_count.items(), key=lambda x: x[1], reverse=True):
    print(f"{cat} | {count}")

Одежда | 1529
Электроника | 1185
Дом | 1105
Красота | 991
Книги | 653


In [ ]:
#3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
months = {}
revenue = {}
count = {}
for order in rows:
    if order["status"] == "оплачен":
        cat = order["category"]
        amount = to_num(order["amount"])
        revenue[cat] = revenue.get(cat, 0) + amount
        count[cat] = count.get(cat, 0) + 1
for cat in revenue:
    avg = revenue[cat] / count[cat]
    print(f"{cat} | {revenue[cat]:.0f} | {avg:.2f}")

Одежда | 18321080 | 13285.77
Красота | 5507620 | 6188.34
Дом | 10516320 | 10654.83
Книги | 2294100 | 3934.99
Электроника | 65270240 | 61286.61


In [ ]:
#4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.
months = {}
for order in rows:
    month = order["order_date"][:7]
    months[month] = months.get(month, 0) + 1
for month, count in sorted(months.items()):
    print(f"{month} | {count}")

2026-09 | 2782
2026-10 | 2681


In [ ]:
## [Стандарт]
#5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
#6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
#7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
#8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [34]:
import numpy as np

In [70]:
#5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
sep = 0.0
oct = 0.0

for order in rows:
    if order["status"] == "оплачен":
        month_val = order["order_date"][:7]
        amount_val = to_num(order["amount"])
        if month_val == "2026-09":
            sep += amount_val
        elif month_val == "2026-10":
            oct += amount_val

print("Сентябрь:", sep)
print("Октябрь:", oct)
print("Изменение:", f"{(oct - sep) / sep:+.1%}")

Сентябрь: 57489850.0
Октябрь: 44419510.0
Изменение: -22.7%


In [76]:
#6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
from collections import defaultdict

total_orders_city = defaultdict(int)
canceled_orders_city = defaultdict(int)

for r in rows:
    try:
        city = clean_city_robust(r['city'])
    except Exception:
        continue

    total_orders_city[city] += 1
    if r['status'].strip().lower() in ('отменён', 'отменен'):
        canceled_orders_city[city] += 1

print("=== 6. Доля отменённых заказов по городам ===")
for city in sorted(total_orders_city.keys()):
    total = total_orders_city[city]
    canceled = canceled_orders_city[city]
    share = (canceled / total) * 100 if total > 0 else 0
    print(f"{city}: {share:.1f}% ({canceled} из {total})")
print()

=== 6. Доля отменённых заказов по городам ===
Актобе: 10.3% (41 из 397)
Алматы: 7.9% (172 из 2188)
Астана: 8.8% (123 из 1390)
Атырау: 7.5% (11 из 146)
Караганда: 8.9% (48 из 538)
Шымкент: 20.4% (164 из 804)



In [74]:
# 7. Топ-5 клиентов по выручке оплаченных заказов ---
client_revenue = {}

for r in rows:
    if r['status'].strip().lower() == 'оплачен':
        cust_id = r['customer_id'].strip()
        amount_val = to_num(r['amount'])
        client_revenue[cust_id] = client_revenue.get(cust_id, 0.0) + amount_val

top_clients = sorted(client_revenue.items(), key=lambda x: x[1], reverse=True)[:5]

print("7. Топ-5 клиентов по выручке оплаченных заказов:")
for cust_id, rev in top_clients:
    print(f"{cust_id}: {rev:.1f}")

7. Топ-5 клиентов по выручке оплаченных заказов:
TEST: 29999970.0
C01627: 257180.0
C00485: 215920.0
C01338: 192080.0
C02122: 191670.0


In [67]:
# --- 8. Сколько уникальных клиентов в каждом городе ---


unique_customers_city = {}

for r in rows:
    try:
        city = clean_city_robust(r['city'])
    except Exception:
        continue

    cust_id = r['customer_id'].strip()

    if city not in unique_customers_city:
        unique_customers_city[city] = set()

    unique_customers_city[city].add(cust_id)

print("8. Уникальные клиенты по городам")
for city in sorted(unique_customers_city.keys()):
    print(f"{city}: {len(unique_customers_city[city])}")

8. Уникальные клиенты по городам
Актобе: 358
Алматы: 1351
Астана: 981
Атырау: 138
Караганда: 451
Шымкент: 655
